In [1]:
!pip install -q haystack-ai pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.8/713.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.0/683.0 kB 44.8 MB/s eta 0:00:00


In [4]:
import haystack
import sys

print("Python version:")
print(sys.version)

print("\nHaystack version:")
print(haystack.__version__)

print("\nHaystack location:")
print(haystack.__file__)

import haystack.components.embedders as embedders

print("\nAvailable embedders:")
print(dir(embedders))

Python version:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Haystack version:
3.1.1

Haystack location:
/usr/local/lib/python3.13/dist-packages/haystack/__init__.py

Available embedders:
['AzureOpenAIDocumentEmbedder', 'AzureOpenAITextEmbedder', 'MockDocumentEmbedder', 'MockTextEmbedder', 'OpenAIDocumentEmbedder', 'OpenAITextEmbedder', '__all__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_exports', '_import_structure', '_name', '_objects', 'azure_document_embedder', 'azure_text_embedder', 'mock_document_embedder', 'mock_text_embedder', 'openai_document_embedder', 'openai_text_embedder']


In [5]:
import sys
import os
from pathlib import Path

import pandas as pd
import numpy as np

import haystack

from haystack import Pipeline
from haystack.dataclasses import Document
from haystack.components.converters import PyPDFToDocument
from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever,
    InMemoryEmbeddingRetriever
)
from haystack.document_stores.in_memory import InMemoryDocumentStore

from sentence_transformers import SentenceTransformer

print("Python version:", sys.version)
print("Haystack version:", haystack.__version__)
print("Sentence Transformers imported successfully!")

print("\n✓ All required imports successful")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Haystack version: 3.1.1
Sentence Transformers imported successfully!

✓ All required imports successful


In [7]:
from google.colab import files

uploaded = files.upload()

pdf_files = list(uploaded.keys())

print("\nUploaded files:")
for file in pdf_files:
    print("-", file)

print("\nNumber of uploaded files:", len(pdf_files))

Saving week7pdf5.pdf to week7pdf5.pdf
Saving week7pdf4.pdf to week7pdf4.pdf
Saving week7pdf3.pdf to week7pdf3.pdf
Saving week7pdf2.pdf to week7pdf2.pdf
Saving week7pdf1.pdf to week7pdf1 (1).pdf

Uploaded files:
- week7pdf5.pdf
- week7pdf4.pdf
- week7pdf3.pdf
- week7pdf2.pdf
- week7pdf1 (1).pdf

Number of uploaded files: 5


In [8]:
pdf_paths = [
    Path(file)
    for file in pdf_files
    if file.lower().endswith(".pdf")
]

if len(pdf_paths) != 5:
    raise ValueError(
        f"Expected exactly 5 PDF files, but found {len(pdf_paths)}. "
        "Please upload exactly 5 PDFs."
    )

print("✓ Exactly 5 PDF files found:\n")

for i, path in enumerate(pdf_paths, start=1):
    print(f"{i}. {path.name}")

✓ Exactly 5 PDF files found:

1. week7pdf5.pdf
2. week7pdf4.pdf
3. week7pdf3.pdf
4. week7pdf2.pdf
5. week7pdf1 (1).pdf


In [9]:
converter = PyPDFToDocument()

conversion_result = converter.run(
    sources=pdf_paths
)

documents = conversion_result["documents"]

print("✓ PDF conversion completed")
print("Total Haystack Documents:", len(documents))

✓ PDF conversion completed
Total Haystack Documents: 5


In [10]:
for i, doc in enumerate(documents[:5], start=1):
    print(f"\n--- Document {i} ---")
    print("Characters:", len(doc.content))
    print("Metadata:", doc.meta)
    print("Preview:")
    print(doc.content[:300].replace("\n", " "))


--- Document 1 ---
Characters: 44162
Metadata: {'file_path': 'week7pdf5.pdf'}
Preview:
Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks Nils Reimers and Iryna Gurevych Ubiquitous Knowledge Processing Lab (UKP-TUDA) Department of Computer Science, Technische Universit¨at Darmstadt www.ukp.tu-darmstadt.de Abstract BERT (Devlin et al., 2018) and RoBERTa (Liu et al., 2019) h

--- Document 2 ---
Characters: 236748
Metadata: {'file_path': 'week7pdf4.pdf'}
Preview:
Language Models are Few-Shot Learners Tom B. Brown∗ Benjamin Mann∗ Nick Ryder∗ Melanie Subbiah∗ Jared Kaplan† Prafulla Dhariwal Arvind Neelakantan Pranav Shyam Girish Sastry Amanda Askell Sandhini Agarwal Ariel Herbert-Voss Gretchen Krueger Tom Henighan Rewon Child Aditya Ramesh Daniel M. Ziegler Je

--- Document 3 ---
Characters: 69076
Metadata: {'file_path': 'week7pdf3.pdf'}
Preview:
Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, V

In [11]:
bm25_store = InMemoryDocumentStore()

print("✓ BM25 document store created")

✓ BM25 document store created


In [12]:
bm25_store.write_documents(documents)

print(
    "✓ Documents indexed:",
    bm25_store.count_documents()
)

✓ Documents indexed: 5


In [13]:
bm25_retriever = InMemoryBM25Retriever(
    document_store=bm25_store,
    top_k=3
)

print("✓ BM25 retriever created")

✓ BM25 retriever created


In [14]:
bm25_pipeline = Pipeline()

bm25_pipeline.add_component(
    "retriever",
    bm25_retriever
)

print("✓ BM25 pipeline created")

✓ BM25 pipeline created


In [15]:
questions = [
    "What is the main objective of the research?",
    "What problem does the research attempt to solve?",
    "What methodology is used in the study?",
    "What dataset is used?",
    "What model or algorithm is proposed?",
    "How is the proposed method evaluated?",
    "What are the main findings of the study?",
    "What are the key experimental results?",
    "What are the limitations of the proposed approach?",
    "What future work is suggested?"
]

print("Total questions:", len(questions))

Total questions: 10


In [16]:
bm25_results = []

for i, question in enumerate(questions, start=1):

    result = bm25_pipeline.run({
        "retriever": {
            "query": question
        }
    })

    retrieved_docs = result["retriever"]["documents"]

    bm25_results.append({
        "question": question,
        "documents": retrieved_docs
    })

    print("\n" + "=" * 80)
    print(f"QUESTION {i}")
    print(question)

    for rank, doc in enumerate(retrieved_docs, start=1):
        print(f"\nRank {rank}")
        print("Score:", doc.score)
        print("Text:", doc.content[:250].replace("\n", " "))


QUESTION 1
What is the main objective of the research?

Rank 1
Score: 3.1192958533739996
Text: Language Models are Few-Shot Learners Tom B. Brown∗ Benjamin Mann∗ Nick Ryder∗ Melanie Subbiah∗ Jared Kaplan† Prafulla Dhariwal Arvind Neelakantan Pranav Shyam Girish Sastry Amanda Askell Sandhini Agarwal Ariel Herbert-Voss Gretchen Krueger Tom Henig

Rank 2
Score: 2.989259051428672
Text: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†, Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡

Rank 3
Score: 2.679015066436271
Text: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com N

QUESTION 2
What problem does the research attempt to solve?

Rank 1
Scor

In [17]:
bm25_table = []

for item in bm25_results:

    for rank, doc in enumerate(item["documents"], start=1):

        bm25_table.append({
            "Question": item["question"],
            "Rank": rank,
            "Score": doc.score,
            "Retrieved Text": doc.content[:200].replace("\n", " ")
        })

bm25_df = pd.DataFrame(bm25_table)

display(bm25_df)

,Question,Rank,Score,Retrieved Text
0,What is the main objective of the research?,1,3.119296,Language Models are Few-Shot Learners Tom B. B...
1,What is the main objective of the research?,2,2.989259,Retrieval-Augmented Generation for Knowledge-I...
2,What is the main objective of the research?,3,2.679015,"Provided proper attribution is provided, Googl..."
3,What problem does the research attempt to solve?,1,6.055119,Language Models are Few-Shot Learners Tom B. B...
4,What problem does the research attempt to solve?,2,4.756376,Retrieval-Augmented Generation for Knowledge-I...
5,What problem does the research attempt to solve?,3,3.732532,"Provided proper attribution is provided, Googl..."
6,What methodology is used in the study?,1,4.888682,Language Models are Few-Shot Learners Tom B. B...
7,What methodology is used in the study?,2,3.125017,"Provided proper attribution is provided, Googl..."
8,What methodology is used in the study?,3,3.091407,Retrieval-Augmented Generation for Knowledge-I...
9,What dataset is used?,1,1.876029,Language Models are Few-Shot Learners Tom B. B...


In [18]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("✓ Sentence Transformer model loaded")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Sentence Transformer model loaded
Embedding dimension: 384


/tmp/ipykernel_522/634569746.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [19]:
document_texts = [
    doc.content
    for doc in documents
]

document_embeddings = embedding_model.encode(
    document_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("✓ Document embeddings generated")
print("Number of embeddings:", len(document_embeddings))
print("Embedding shape:", document_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Document embeddings generated
Number of embeddings: 5
Embedding shape: (5, 384)


In [20]:
embedded_documents = []

for doc, embedding in zip(documents, document_embeddings):

    embedded_doc = Document(
        content=doc.content,
        meta=doc.meta,
        embedding=embedding.tolist()
    )

    embedded_documents.append(embedded_doc)

print("✓ Embeddings attached to Haystack Documents")
print("Documents:", len(embedded_documents))

✓ Embeddings attached to Haystack Documents
Documents: 5


In [21]:
dense_store = InMemoryDocumentStore(
    embedding_similarity_function="cosine"
)

print("✓ Dense document store created")

✓ Dense document store created


In [22]:
dense_store.write_documents(
    embedded_documents
)

print(
    "✓ Embedded documents indexed:",
    dense_store.count_documents()
)

✓ Embedded documents indexed: 5


In [23]:
dense_retriever = InMemoryEmbeddingRetriever(
    document_store=dense_store,
    top_k=3
)

print("✓ Dense retriever created")

✓ Dense retriever created


In [24]:
from haystack import component


@component
class QueryEmbedder:

    @component.output_types(query_embedding=list)

    def run(self, query: str):
        embedding = embedding_model.encode(
            query,
            convert_to_numpy=True
        )

        return {
            "query_embedding": embedding.tolist()
        }


query_embedder = QueryEmbedder()

print("✓ Query embedder created")

✓ Query embedder created


In [31]:
from haystack import Pipeline, component

@component
class DenseQueryEmbedder:

    @component.output_types(query_embedding=list[float])
    def run(self, query: str):

        embedding = embedding_model.encode(
            query,
            convert_to_numpy=True
        )

        return {
            "query_embedding": embedding.tolist()
        }


# Create a fresh query embedder
dense_query_embedder = DenseQueryEmbedder()

# Create a fresh retriever
dense_retriever_new = InMemoryEmbeddingRetriever(
    document_store=dense_store,
    top_k=3
)

# Create a fresh pipeline
dense_pipeline = Pipeline()

dense_pipeline.add_component(
    "query_embedder",
    dense_query_embedder
)

dense_pipeline.add_component(
    "retriever",
    dense_retriever_new
)

dense_pipeline.connect(
    "query_embedder.query_embedding",
    "retriever.query_embedding"
)

print("✓ Dense retrieval pipeline created successfully")

✓ Dense retrieval pipeline created successfully


In [32]:
dense_results = []

for i, question in enumerate(questions, start=1):

    result = dense_pipeline.run({
        "query_embedder": {
            "query": question
        }
    })

    retrieved_docs = result["retriever"]["documents"]

    dense_results.append({
        "question": question,
        "documents": retrieved_docs
    })

    print("\n" + "=" * 80)
    print(f"QUESTION {i}")
    print(question)

    for rank, doc in enumerate(retrieved_docs, start=1):
        print(f"\nRank {rank}")
        print("Score:", doc.score)
        print("Text:", doc.content[:250].replace("\n", " "))


QUESTION 1
What is the main objective of the research?

Rank 1
Score: 0.054528683253088926
Text: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com N

Rank 2
Score: 0.01921314832664016
Text: Language Models are Few-Shot Learners Tom B. Brown∗ Benjamin Mann∗ Nick Ryder∗ Melanie Subbiah∗ Jared Kaplan† Prafulla Dhariwal Arvind Neelakantan Pranav Shyam Girish Sastry Amanda Askell Sandhini Agarwal Ariel Herbert-Voss Gretchen Krueger Tom Henig

Rank 3
Score: 0.01706660417482726
Text: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†, Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡

QUESTION 2
What problem does the research attempt to solve?

Rank 

In [33]:
dense_table = []

for item in dense_results:

    for rank, doc in enumerate(item["documents"], start=1):

        dense_table.append({
            "Question": item["question"],
            "Rank": rank,
            "Score": doc.score,
            "Retrieved Text": doc.content[:200].replace("\n", " ")
        })

dense_df = pd.DataFrame(dense_table)

display(dense_df)

,Question,Rank,Score,Retrieved Text
0,What is the main objective of the research?,1,0.054529,"Provided proper attribution is provided, Googl..."
1,What is the main objective of the research?,2,0.019213,Language Models are Few-Shot Learners Tom B. B...
2,What is the main objective of the research?,3,0.017067,Retrieval-Augmented Generation for Knowledge-I...
3,What problem does the research attempt to solve?,1,0.077273,Language Models are Few-Shot Learners Tom B. B...
4,What problem does the research attempt to solve?,2,0.075739,"Provided proper attribution is provided, Googl..."
5,What problem does the research attempt to solve?,3,0.060080,Retrieval-Augmented Generation for Knowledge-I...
6,What methodology is used in the study?,1,0.053395,Language Models are Few-Shot Learners Tom B. B...
7,What methodology is used in the study?,2,0.021616,"Provided proper attribution is provided, Googl..."
8,What methodology is used in the study?,3,0.016776,Retrieval-Augmented Generation for Knowledge-I...
9,What dataset is used?,1,0.172070,Language Models are Few-Shot Learners Tom B. B...


In [5]:
questions = [
    "What is the main objective of the research?",
    "What problem does the research attempt to solve?",
    "What methodology is used in the study?",
    "What dataset is used?",
    "What model or algorithm is proposed?",
    "How is the proposed method evaluated?",
    "What are the main findings of the study?",
    "What are the key experimental results?",
    "What are the limitations of the proposed approach?",
    "What future work is suggested?"
]

print("✓ Questions loaded:", len(questions))

✓ Questions loaded: 10


In [6]:
# Cell 24 - Manual Evaluation

evaluation = []

print("Enter 1 if the TOP retrieved document is relevant.")
print("Enter 0 if the TOP retrieved document is not relevant.")

for i, question in enumerate(questions, start=1):

    print("\n" + "=" * 70)
    print(f"Question {i}:")
    print(question)

    bm25_value = int(input("BM25 top result relevant? (1/0): "))
    dense_value = int(input("Dense top result relevant? (1/0): "))

    evaluation.append({
        "Question": question,
        "BM25": bm25_value,
        "Dense": dense_value
    })

print("\n✓ Evaluation completed!")



Enter 1 if the TOP retrieved document is relevant.
Enter 0 if the TOP retrieved document is not relevant.

Question 1:
What is the main objective of the research?
BM25 top result relevant? (1/0): 1
Dense top result relevant? (1/0): 1

Question 2:
What problem does the research attempt to solve?
BM25 top result relevant? (1/0): 1
Dense top result relevant? (1/0): 1

Question 3:
What methodology is used in the study?
BM25 top result relevant? (1/0): 1
Dense top result relevant? (1/0): 1

Question 4:
What dataset is used?
BM25 top result relevant? (1/0): 0
Dense top result relevant? (1/0): 0

Question 5:
What model or algorithm is proposed?
BM25 top result relevant? (1/0): 1
Dense top result relevant? (1/0): 1

Question 6:
How is the proposed method evaluated?
BM25 top result relevant? (1/0): 1
Dense top result relevant? (1/0): 1

Question 7:
What are the main findings of the study?
BM25 top result relevant? (1/0): 0
Dense top result relevant? (1/0): 0

Question 8:
What are the key experi

In [8]:
import pandas as pd

print("✓ Pandas loaded successfully")

✓ Pandas loaded successfully


In [9]:
evaluation_df = pd.DataFrame(evaluation)

bm25_relevant = evaluation_df["BM25"].sum()
dense_relevant = evaluation_df["Dense"].sum()

bm25_rate = evaluation_df["BM25"].mean()
dense_rate = evaluation_df["Dense"].mean()

print("BM25 relevant results:", bm25_relevant, "/ 10")
print("Dense relevant results:", dense_relevant, "/ 10")

print("\nBM25 relevance rate:", round(bm25_rate * 100, 2), "%")
print("Dense relevance rate:", round(dense_rate * 100, 2), "%")

BM25 relevant results: 6 / 10
Dense relevant results: 6 / 10

BM25 relevance rate: 60.0 %
Dense relevance rate: 60.0 %


In [10]:
comparison = pd.DataFrame({
    "Method": [
        "BM25",
        "Dense Retrieval"
    ],
    "Questions Tested": [
        10,
        10
    ],
    "Top-K": [
        3,
        3
    ],
    "Relevant Top Results": [
        bm25_relevant,
        dense_relevant
    ],
    "Relevance Rate (%)": [
        round(bm25_rate * 100, 2),
        round(dense_rate * 100, 2)
    ]
})

display(comparison)

,Method,Questions Tested,Top-K,Relevant Top Results,Relevance Rate (%)
0,BM25,10,3,6,60.0
1,Dense Retrieval,10,3,6,60.0


In [11]:
display(evaluation_df)

,Question,BM25,Dense
0,What is the main objective of the research?,1,1
1,What problem does the research attempt to solve?,1,1
2,What methodology is used in the study?,1,1
3,What dataset is used?,0,0
4,What model or algorithm is proposed?,1,1
5,How is the proposed method evaluated?,1,1
6,What are the main findings of the study?,0,0
7,What are the key experimental results?,1,1
8,What are the limitations of the proposed appro...,0,0
9,What future work is suggested?,0,0


In [12]:
evaluation_df.to_csv(
    "haystack_bm25_vs_dense_results.csv",
    index=False
)

comparison.to_csv(
    "haystack_comparison_summary.csv",
    index=False
)

print("✓ Results saved successfully")

✓ Results saved successfully
